# CSE Employability Skill-Gap Analysis using Preprocessing and Feast

## Complete End-to-End Notebook

**Project:** Employability Skill-Gap Analysis for CSE Graduates

### Workflow

**Dataset → Preprocessing → Machine Learning → Feast Feature Store → Historical Feature Retrieval → Online Feature Retrieval → Prediction**

This single notebook combines the **preprocessing** and **Feast** parts so that everything can be executed from one place.

### Project Objective

The project analyses CSE graduate skills by comparing curriculum-based skill scores with industry requirements and classifying students into:

- Low Skill Gap
- Medium Skill Gap
- High Skill Gap


# 1. Install Required Libraries

The notebook uses:

- Pandas
- NumPy
- Scikit-learn
- Joblib
- PyArrow
- Feast


In [ ]:
!pip -q install pandas numpy scikit-learn joblib pyarrow feast==0.64.0


# 2. Import Libraries


In [ ]:
import os
import shutil
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import feast
from feast import FeatureStore

print("Feast version:", feast.__version__)


# 3. Load the Newly Generated Dataset

Upload the unique dataset:

`CSE_Employability_Skill_Gap_Unique_Dataset.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()

data = pd.read_csv("CSE_Employability_Skill_Gap_Unique_Dataset.csv")

print("Dataset shape:", data.shape)
display(data.head())


# 4. Dataset Understanding and Initial Inspection


In [ ]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)

print("\nMissing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())
print("Duplicate Student IDs:", data["Student_ID"].duplicated().sum())


# 5. Data Cleaning

Basic cleaning is performed before preprocessing:

1. Remove duplicate rows.
2. Remove duplicate Student IDs.
3. Standardize categorical text values.


In [ ]:
data = data.drop_duplicates().copy()
data = data.drop_duplicates(subset=["Student_ID"], keep="first").copy()

for col in ["Skill_Gap_Category", "Priority_Training_Area"]:
    data[col] = data[col].astype(str).str.strip()

print("Cleaned dataset shape:", data.shape)


# 6. Define Features and Target

### Target

`Skill_Gap_Category`

### Important

The dataset contains calculated gap-related columns. These are excluded from model features to avoid **data leakage**, because directly using the calculated gap would make the classification artificially easy.


In [ ]:
target = "Skill_Gap_Category"

excluded = {
    "Student_ID",
    target,
    "Overall_Gap_Score",
    "Priority_Training_Area",
    "Employability_Readiness"
}

# Exclude all individual calculated gap columns.
excluded.update([
    c for c in data.columns
    if c.endswith("_Gap")
])

feature_cols = [
    c for c in data.columns
    if c not in excluded
]

X = data[feature_cols].copy()
y = data[target].copy()

print("Number of input features:", len(feature_cols))
print("\nInput features:")
for col in feature_cols:
    print("-", col)

print("\nTarget distribution:")
print(y.value_counts())


# 7. Identify Numerical and Categorical Features


In [ ]:
categorical_features = [
    c for c in feature_cols
    if X[c].dtype == "object"
]

numerical_features = [
    c for c in feature_cols
    if c not in categorical_features
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


# 8. Train-Test Split

The dataset is divided into:

- 80% Training data
- 20% Testing data

Stratification is used so the Low/Medium/High categories remain represented in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


# 9. Numerical Data Preprocessing

### Steps

1. Missing values → median
2. Feature scaling → StandardScaler


In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# 10. Categorical Data Preprocessing

### Steps

1. Missing values → most frequent value
2. Categorical values → One-Hot Encoding


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# 11. Combine Preprocessing Pipelines


In [ ]:
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# 12. Machine Learning Model

A basic **Logistic Regression** classifier is used, matching the project's requirement to apply one basic machine-learning model.

The complete pipeline performs preprocessing first and then classification.


In [ ]:
ml_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

ml_pipeline.fit(X_train, y_train)

predictions = ml_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, predictions))


# 13. Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, predictions)

cm_df = pd.DataFrame(
    cm,
    index=["Actual High", "Actual Low", "Actual Medium"],
    columns=["Pred High", "Pred Low", "Pred Medium"]
)

display(cm_df)


# 14. Generate the Preprocessed Dataset

The fitted preprocessing transformer is used to convert the raw input features into a numerical model-ready representation.


In [ ]:
X_train_processed = ml_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = ml_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

processed_feature_names = ml_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

processed_train = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

processed_test = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

processed_train[target] = y_train.values
processed_test[target] = y_test.values

print("Processed training shape:", processed_train.shape)
print("Processed testing shape:", processed_test.shape)

display(processed_train.head())


# 15. Save Preprocessed Data


In [ ]:
processed_train.to_csv(
    "CSE_employability_processed_train.csv",
    index=False
)

processed_test.to_csv(
    "CSE_employability_processed_test.csv",
    index=False
)

joblib.dump(
    ml_pipeline,
    "CSE_employability_preprocessing_model_pipeline.pkl"
)

print("Preprocessed files and ML pipeline saved.")


# ============================================================
# 16. FEAST FEATURE STORE
# ============================================================

## Purpose of Feast

Feast is used to manage and retrieve machine-learning features consistently.

In this project:

**Raw Dataset → Preprocessing → Feature Data → Feast Feature Store → Historical/Online Features → ML Prediction**


# 17. Prepare Data for Feast

Feast needs an entity identifier and timestamps.

For this project:

- Entity = `student_id`
- Event timestamp = generated timestamp
- Features = curriculum/industry skill measurements and academic indicators


In [ ]:
# Create Feast-compatible feature data.
feast_df = data.copy()

feast_df["student_id"] = feast_df["Student_ID"].astype(str)

# Deterministic timestamps for the feature-store demonstration.
base_time = pd.Timestamp("2026-01-01", tz="UTC")

feast_df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        np.arange(len(feast_df)),
        unit="s"
    )
)

feast_df["created_timestamp"] = (
    feast_df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

# Features used by Feast.
feast_feature_columns = [
    "student_id",
    "event_timestamp",
    "created_timestamp",
    "Year",
    "Semester",
    "Attendance_Percent",
    "Projects_Completed",
    "Certifications",
    "Programming_Curriculum",
    "Programming_Industry",
    "Database_Curriculum",
    "Database_Industry",
    "Cloud_Curriculum",
    "Cloud_Industry",
    "DataAnalysis_Curriculum",
    "DataAnalysis_Industry",
    "ProblemSolving_Curriculum",
    "ProblemSolving_Industry",
    "Communication_Curriculum",
    "Communication_Industry",
    "Teamwork_Curriculum",
    "Teamwork_Industry",
    "Aptitude_Curriculum",
    "Aptitude_Industry"
]

feast_features = feast_df[feast_feature_columns].copy()

display(feast_features.head())


# 18. Create Feast Repository


In [ ]:
repo_path = "/content/cse_employability_feast"

# Recreate repository for a clean run.
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

os.makedirs(f"{repo_path}/data", exist_ok=True)

feast_features.to_parquet(
    f"{repo_path}/data/student_features.parquet",
    index=False
)

print("Feast repository created at:", repo_path)


# 19. Create `feature_store.yaml`


In [ ]:
feature_store_config = '''
project: cse_employability_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
'''

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_config)

print("feature_store.yaml created.")


# 20. Define Feast Entity, Feature View and Feature Service


In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import Float32, Int64, String

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE graduate student"
)

student_source = FileSource(
    name="student_skill_source",
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="Year", dtype=Int64),
        Field(name="Semester", dtype=Int64),
        Field(name="Attendance_Percent", dtype=Float32),
        Field(name="Projects_Completed", dtype=Int64),
        Field(name="Certifications", dtype=Int64),

        Field(name="Programming_Curriculum", dtype=Float32),
        Field(name="Programming_Industry", dtype=Float32),

        Field(name="Database_Curriculum", dtype=Float32),
        Field(name="Database_Industry", dtype=Float32),

        Field(name="Cloud_Curriculum", dtype=Float32),
        Field(name="Cloud_Industry", dtype=Float32),

        Field(name="DataAnalysis_Curriculum", dtype=Float32),
        Field(name="DataAnalysis_Industry", dtype=Float32),

        Field(name="ProblemSolving_Curriculum", dtype=Float32),
        Field(name="ProblemSolving_Industry", dtype=Float32),

        Field(name="Communication_Curriculum", dtype=Float32),
        Field(name="Communication_Industry", dtype=Float32),

        Field(name="Teamwork_Curriculum", dtype=Float32),
        Field(name="Teamwork_Industry", dtype=Float32),

        Field(name="Aptitude_Curriculum", dtype=Float32),
        Field(name="Aptitude_Industry", dtype=Float32),
    ],
    source=student_source,
    online=True
)

employability_service = FeatureService(
    name="cse_employability_service",
    features=[student_skill_features]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

print("Feast feature definitions created.")


# 21. Apply Feast Definitions

This registers the entity, feature view and feature service.


In [ ]:
%cd /content/cse_employability_feast

!feast apply


# 22. Verify Feast Objects


In [ ]:
!feast entities list
!feast feature-views list


# 23. Create Feast Store Object


In [ ]:
store = FeatureStore(
    repo_path=repo_path
)

feature_service = store.get_feature_service(
    "cse_employability_service"
)

print("Feast Feature Store ready.")


# 24. Historical Feature Retrieval

Historical retrieval is useful for building machine-learning training data without accidentally using future feature values.

The student's timestamp is used as the point-in-time reference.


In [ ]:
entity_df = feast_df[
    ["student_id", "event_timestamp"]
].copy()

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

print("Historical feature shape:", historical_features.shape)
display(historical_features.head())


# 25. Materialize Features to the Online Store


In [ ]:
%cd /content/cse_employability_feast

start_time = feast_features["event_timestamp"].min().strftime(
    "%Y-%m-%dT%H:%M:%S"
)

end_time = (
    feast_features["event_timestamp"].max()
    + pd.Timedelta(minutes=1)
).strftime("%Y-%m-%dT%H:%M:%S")

print("Materializing from:", start_time)
print("Materializing to:", end_time)

!feast materialize $start_time $end_time


# 26. Online Feature Retrieval

Online retrieval demonstrates how the latest stored features can be retrieved for a student for real-time prediction.


In [ ]:
sample_student_id = feast_features.iloc[0]["student_id"]

online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": sample_student_id}
    ]
).to_dict()

online_df = pd.DataFrame(online_features)

display(online_df)


# 27. Use Feast Features for Prediction

The online Feast features are converted into the same input structure expected by the trained model.

For demonstration, the prediction is made using the corresponding student's stored skill information.


In [ ]:
# Build a prediction row from the original feature values.
prediction_row = feast_df[
    feast_df["student_id"] == sample_student_id
][feature_cols].copy()

prediction = ml_pipeline.predict(prediction_row)

print("Student ID:", sample_student_id)
print("Predicted Skill-Gap Category:", prediction[0])


# 28. Retrieve Multiple Students from Feast


In [ ]:
sample_ids = feast_features.iloc[
    :5
]["student_id"].tolist()

multi_online = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": sid}
        for sid in sample_ids
    ]
).to_dict()

multi_online_df = pd.DataFrame(multi_online)

display(multi_online_df)


# 29. Final Project Pipeline Summary

### Complete workflow implemented in this notebook

1. **Load unique synthetic CSE dataset**
2. **Inspect dataset**
3. **Clean duplicate records**
4. **Separate features and target**
5. **Split training and testing data**
6. **Handle missing numerical values**
7. **Handle missing categorical values**
8. **Standardize numerical features**
9. **One-hot encode categorical features**
10. **Train Logistic Regression**
11. **Evaluate model**
12. **Generate preprocessed datasets**
13. **Create Feast repository**
14. **Define Feast Entity**
15. **Define Feast Feature View**
16. **Define Feast Feature Service**
17. **Apply Feast configuration**
18. **Retrieve historical features**
19. **Materialize features**
20. **Retrieve online features**
21. **Perform prediction**

### Final Architecture

**Unique Dataset**
↓  
**Preprocessing**
↓  
**Machine Learning Model**
↓  
**Feast Feature Store**
↓  
**Historical / Online Feature Retrieval**
↓  
**Skill-Gap Prediction**


# 30. Download the Generated Outputs

The following files can be downloaded after running the notebook:

- Processed training dataset
- Processed testing dataset
- Saved preprocessing + ML pipeline


In [ ]:
from google.colab import files

files.download("CSE_employability_processed_train.csv")
files.download("CSE_employability_processed_test.csv")
files.download("CSE_employability_preprocessing_model_pipeline.pkl")
